 # Chapter 2 – Sector Proxy vs Stock Representativeness

 - Produces:

   - UMAP cluster plot (intra-sector structure),

   - Sector proxy vs representative stock (normalized index),

   - Tracking error comparison (single stock and Top-5 basket).

In [1]:
from pathlib import Path
import pandas as pd
import plotly.io as pio
from pathlib import Path
from typing import Literal

import chapter5 as ch5

pio.renderers.default = "notebook"  # change to "vscode" if preferred


 ## Load parquet data (stocks_sect_clean)

In [2]:
data = ch5.load_data(Path("data_processed"), stocks_file="stocks_sect_clean.parquet")
df_stock_sect = data["stocks_sect"]
df_stock_sect.head()


,Date,Weekday,Open,High,Low,Close,Adj Close,Volume,Symbol,Volatility,ticker,Sector,Return,SectorVolume,Weight,RollingVol60,RollingVol120,WeightedVol60,WeightedVol120
0,1999-11-18,3,32.546494,35.765381,28.612303,31.473534,27.068665,62546300.0,A,0.219780,A,Healthcare,NaN,2.431992e+08,0.257181,NaN,NaN,NaN,NaN
1,1999-11-19,4,30.713520,30.758226,28.478184,28.880543,24.838577,15234100.0,A,0.074236,A,Healthcare,-0.085979,1.326396e+08,0.114853,NaN,NaN,NaN,NaN
2,1999-11-22,0,29.551144,31.473534,28.657009,31.473534,27.068665,6577800.0,A,0.095310,A,Healthcare,0.085979,1.350682e+08,0.048700,NaN,NaN,NaN,NaN
3,1999-11-23,1,30.400572,31.205294,28.612303,28.612303,24.607880,5975600.0,A,0.085294,A,Healthcare,-0.095310,1.290555e+08,0.046303,NaN,NaN,NaN,NaN
4,1999-11-24,2,28.701717,29.998211,28.612303,29.372318,25.261524,4843200.0,A,0.048287,A,Healthcare,0.026216,1.125289e+08,0.043040,NaN,NaN,NaN,NaN


 ## List available sectors

In [3]:
sectors = sorted(df_stock_sect["Sector"].dropna().unique())
print(f"Found {len(sectors)} sectors:")
print(sectors)


Found 11 sectors:
['Basic Materials', 'Communication Services', 'Consumer Cyclical', 'Consumer Defensive', 'Energy', 'Financial Services', 'Healthcare', 'Industrials', 'Real Estate', 'Technology', 'Utilities']


 ## Helper to run a sector

In [4]:
def analyze_sector(
    df_stock_sect: pd.DataFrame,
    sector_name: str,
    *,
    # --- Returns & representativeness ---
    return_col: str = "Return",
    window_rep: int = 90,
    lifetime_alpha: float = 2.0,
    min_lifetime_fraction: float | None = None,
    
    # --- Tracking error ---
    te_windows: tuple[int, int] = (60, 120),
    te_y_range: tuple[float, float] | None = None,
    # --- Correlation computation ---
    min_overlap_corr: int = 30,
    # --- Distance construction ---
    distance_scale: float = 0.3,
    distance_metric: Literal["linear", "angular"] = "angular",
    nan_distance_fill: Literal["max", "p90", "median"] = "max",
    # --- Missing-data handling ---
    max_nan_frac: float = 0.2,
    # --- Clustering ---
    cluster_method: Literal["agglomerative", "hdbscan", "hdbscan_umap"] = "hdbscan_umap",
    cluster_n: int | None = None,
    cluster_distance_threshold: float | None = None,
    linkage: Literal["average", "complete", "single"] = "complete",
    threshold_percentile: float = 30.0,
    # --- HDBSCAN specific ---
    hdbscan_min_cluster_size: int = 5,
    hdbscan_min_samples: int | None = None,
    # --- UMAP ---
    umap_n_neighbors: int = 10,
    umap_min_dist: float = 0.01,
    # --- Visualization of MissingData ---
    missingdata_spread: float = 5.0,
    missingdata_jitter: float = 0.2,
) -> dict:
    """
    Run the full intra-sector representativeness and clustering analysis
    for a single sector.
    Lifetime-weighted representativeness penalizes very recent stocks so the
    Top basket holds names with enough history for stable tracking.
    """
    return ch5.run_sector_analysis(
        df_stock_sect=df_stock_sect,
        sector=sector_name,
        return_col=return_col,
        window_rep=window_rep,
        te_windows=te_windows,
        min_overlap_corr=min_overlap_corr,
        distance_scale=distance_scale,
        cluster_method=cluster_method,
        cluster_n=cluster_n,
        cluster_distance_threshold=cluster_distance_threshold,
        linkage=linkage,
        threshold_percentile=threshold_percentile,
        distance_metric=distance_metric,
        nan_distance_fill=nan_distance_fill,
        max_nan_frac=max_nan_frac,
        hdbscan_min_cluster_size=hdbscan_min_cluster_size,
        hdbscan_min_samples=hdbscan_min_samples,
        missingdata_spread=missingdata_spread,
        missingdata_jitter=missingdata_jitter,
        umap_n_neighbors=umap_n_neighbors,
        umap_min_dist=umap_min_dist,
        lifetime_alpha=lifetime_alpha,
        min_lifetime_fraction=min_lifetime_fraction,
        te_y_range=te_y_range,
    )


 ## Single-sector analysis

 Set the sector you want to analyze.

In [5]:
SECTOR_NAME = "Energy"  # change as needed
TE_Y_RANGE = (0, 0.5)  # shared TE axis range across sectors; adjust as needed
results = analyze_sector(df_stock_sect, SECTOR_NAME, min_overlap_corr=2, distance_scale=2.0, te_y_range=TE_Y_RANGE)
fig_umap = results["fig_umap"]
fig_index = results["fig_index"]
fig_te = results["fig_te"]
if fig_umap is not None:
    fig_umap = ch5.apply_blog_layout(fig_umap, theme="dark", width=500, height=250)
    fig_umap.show()
fig_index = ch5.apply_blog_layout(fig_index, theme="dark", width=500, height=250)
fig_index.show()
fig_te = ch5.apply_blog_layout(fig_te, theme="dark", width=500, height=250)
fig_te.show()


/Users/hugo/miniconda3/envs/py/lib/python3.11/site-packages/umap/umap_.py:1865: UserWarning:

using precomputed metric; inverse_transform will be unavailable

/Users/hugo/miniconda3/envs/py/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.

OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.
/Users/hugo/miniconda3/envs/py/lib/python3.11/site-packages/umap/umap_.py:1865: UserWarning:

using precomputed metric; inverse_transform will be unavailable

/Users/hugo/miniconda3/envs/py/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.

/Users/hugo/miniconda3/envs/py/lib/python3.11/site-packages/sklearn/utils/deprecation.py:132: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

/Users/hugo/miniconda3/en

 ## Key tables and metrics

In [6]:
scores_df = results["scores"]
risk_df = results["risk"]
lifetime_fraction = results.get("lifetime_fraction", pd.Series())
effective_score = results.get("effective_score", pd.Series())
basket_tickers = results["top_tickers"]
single_ticker = results["single_ticker"]

print("Lifetime alpha:", results.get("lifetime_alpha"))
print("Best representative stock:", single_ticker)
print("Top-5 basket:", basket_tickers)

display(
    scores_df
    .assign(lifetime_fraction=lifetime_fraction, effective_score=effective_score)
    .sort_values("effective_score", ascending=False)
    .head(10)
)
if basket_tickers:
    display(risk_df.loc[basket_tickers])
    print("Basket lifetime fractions:", lifetime_fraction[basket_tickers].to_dict())


Lifetime alpha: 2.0
Best representative stock: XOM
Top-5 basket: ['XOM', 'CVX', 'BP', 'SLB', 'NBR']


,rep_score,avg_corr,n_obs,lifetime_fraction,effective_score
Symbol,,,,,
XOM,0.756688,0.756688,14622,0.999932,0.756585
CVX,0.654714,0.654714,14622,0.999932,0.654625
BP,0.591102,0.591102,10867,0.743828,0.327045
SLB,0.694187,0.694187,9605,0.657755,0.300334
NBR,0.454640,0.454640,11843,0.810394,0.298579
COP,0.666184,0.666184,9605,0.657755,0.288219
APA,0.579256,0.579256,10270,0.703110,0.286363
MUR,0.566388,0.566388,10057,0.688583,0.268551
HP,0.580511,0.580511,9910,0.678557,0.267290


,vol,max_drawdown,es_alpha
XOM,0.014155,0.675742,-0.031114
CVX,0.016139,0.618909,-0.035833
BP,0.017850,0.784979,-0.040416
SLB,0.022174,0.941273,-0.049771
NBR,0.048058,1.000000,-0.108328


Basket lifetime fractions: {'XOM': 0.9999317964806984, 'CVX': 0.9999317964806984, 'BP': 0.7438275815032056, 'SLB': 0.6577547401445915, 'NBR': 0.8103942163415633}


 ## Loop over all sectors

 Set `SAVE_FIGS=True` to write HTML files for each plot.

In [7]:
EXPORT_DIR = Path("plots_export")
WEBSITE_EXPORT_DIR = Path("website/chapter_5_plots/plots")
IFRAME_HEIGHT = 560
SAVE_FIGS = True
TE_Y_RANGE = (0, 0.5)  # shared TE axis range across sectors; adjust as needed
exported_files = []
exported_files_website = []
for sect in sectors:
    print(f"=== {sect} ===")
    res = analyze_sector(df_stock_sect, sect, min_overlap_corr=2, distance_scale=2.0, te_y_range=TE_Y_RANGE)
    fu, fi, ft = res["fig_umap"], res["fig_index"], res["fig_te"]
    if fu is not None:
        fu = ch5.apply_blog_layout(fu, theme="dark", width=500, height=250)
        fu.show()
    fi = ch5.apply_blog_layout(fi, theme="dark", width=500, height=250)
    ft = ch5.apply_blog_layout(ft, theme="dark", width=500, height=250)
    fi.show()
    ft.show()
    if SAVE_FIGS:
        prefix = sect.replace(" ", "_").lower()
        exported = ch5.export_sector_figures(res, EXPORT_DIR, prefix, theme="dark")
        exported_files.extend(exported)
        # also export into website folder for iframe consumption
        exported_site = ch5.export_sector_figures(res, WEBSITE_EXPORT_DIR, prefix, theme="dark")
        exported_files_website.extend(exported_site)
if SAVE_FIGS and exported_files:
    snippet_path = ch5.write_iframe_snippets(EXPORT_DIR, exported_files, iframe_height=IFRAME_HEIGHT)
    print(f"Exported {len(exported_files)} files to {EXPORT_DIR} (snippets -> {snippet_path})")
if SAVE_FIGS and exported_files_website:
    print(f"Also exported {len(exported_files_website)} files to {WEBSITE_EXPORT_DIR} for the website Chapter 5 section.")


=== Basic Materials ===


/Users/hugo/miniconda3/envs/py/lib/python3.11/site-packages/umap/umap_.py:1865: UserWarning:

using precomputed metric; inverse_transform will be unavailable

/Users/hugo/miniconda3/envs/py/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.

/Users/hugo/miniconda3/envs/py/lib/python3.11/site-packages/umap/umap_.py:1865: UserWarning:

using precomputed metric; inverse_transform will be unavailable

/Users/hugo/miniconda3/envs/py/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.

/Users/hugo/miniconda3/envs/py/lib/python3.11/site-packages/sklearn/utils/deprecation.py:132: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

/Users/hugo/miniconda3/envs/py/lib/python3.11/site-packages/sklearn/utils/deprecation.py:132: FutureWarning:

'force_all_f

=== Communication Services ===


/Users/hugo/miniconda3/envs/py/lib/python3.11/site-packages/umap/umap_.py:1865: UserWarning:

using precomputed metric; inverse_transform will be unavailable

/Users/hugo/miniconda3/envs/py/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.

/Users/hugo/miniconda3/envs/py/lib/python3.11/site-packages/umap/umap_.py:1865: UserWarning:

using precomputed metric; inverse_transform will be unavailable

/Users/hugo/miniconda3/envs/py/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.

/Users/hugo/miniconda3/envs/py/lib/python3.11/site-packages/sklearn/utils/deprecation.py:132: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

/Users/hugo/miniconda3/envs/py/lib/python3.11/site-packages/sklearn/utils/deprecation.py:132: FutureWarning:

'force_all_f

=== Consumer Cyclical ===


/Users/hugo/miniconda3/envs/py/lib/python3.11/site-packages/umap/umap_.py:1865: UserWarning:

using precomputed metric; inverse_transform will be unavailable

/Users/hugo/miniconda3/envs/py/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.

/Users/hugo/miniconda3/envs/py/lib/python3.11/site-packages/umap/umap_.py:1865: UserWarning:

using precomputed metric; inverse_transform will be unavailable

/Users/hugo/miniconda3/envs/py/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.

/Users/hugo/miniconda3/envs/py/lib/python3.11/site-packages/sklearn/utils/deprecation.py:132: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

/Users/hugo/miniconda3/envs/py/lib/python3.11/site-packages/sklearn/utils/deprecation.py:132: FutureWarning:

'force_all_f

... (13 outputs left out)

/Users/hugo/miniconda3/envs/py/lib/python3.11/site-packages/umap/umap_.py:1865: UserWarning:

using precomputed metric; inverse_transform will be unavailable

/Users/hugo/miniconda3/envs/py/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.

/Users/hugo/miniconda3/envs/py/lib/python3.11/site-packages/umap/umap_.py:1865: UserWarning:

using precomputed metric; inverse_transform will be unavailable

/Users/hugo/miniconda3/envs/py/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.

/Users/hugo/miniconda3/envs/py/lib/python3.11/site-packages/sklearn/utils/deprecation.py:132: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

/Users/hugo/miniconda3/envs/py/lib/python3.11/site-packages/sklearn/utils/deprecation.py:132: FutureWarning:

'force_all_f

=== Utilities ===


/Users/hugo/miniconda3/envs/py/lib/python3.11/site-packages/umap/umap_.py:1865: UserWarning:

using precomputed metric; inverse_transform will be unavailable

/Users/hugo/miniconda3/envs/py/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.

/Users/hugo/miniconda3/envs/py/lib/python3.11/site-packages/umap/umap_.py:1865: UserWarning:

using precomputed metric; inverse_transform will be unavailable

/Users/hugo/miniconda3/envs/py/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.

/Users/hugo/miniconda3/envs/py/lib/python3.11/site-packages/sklearn/utils/deprecation.py:132: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

/Users/hugo/miniconda3/envs/py/lib/python3.11/site-packages/sklearn/utils/deprecation.py:132: FutureWarning:

'force_all_f

Exported 33 files to plots_export (snippets -> plots_export/iframe_snippets.txt)
Also exported 33 files to website/chapter_5_plots/plots for the website Chapter 5 section.


## Basic Materials

In the UMAP, the sector splits into **2 clusters** over **128 stocks**. The structure is less collapsed than most sectors: the largest cluster (**Cluster 1 (94 stocks)**) contains **94 stocks**, i.e. **73.4%** of the sector.

On the index comparison, the **single representative** (**AA**) reaches a return correlation to the proxy of **0.681** with an **annualized tracking error** of **0.262**. The **basket** ((AA, DD, OLN, NUE, PPG)) increases correlation to **0.801** and reduces annualized tracking error to **0.174**.

The rolling TE (60d) confirms the improvement: the single has a **p95 TE** of **0.028** (max **0.047**), while the basket has a **p95 TE** of **0.021** (max **0.035**).

The single stock shows substantially higher tracking error than the basket. This confirms strong internal diversification driven by different materials and value chains, making single-stock representation inadequate.


## Communication Services

In the UMAP, the sector splits into **2 clusters** over **117 stocks**. The structure is concentrated: the largest cluster (**Cluster 1 (105 stocks)**) contains **105 stocks**, i.e. **89.7%** of the sector.

On the index comparison, the **single representative** (**DIS**) reaches a return correlation to the proxy of **0.674** with an **annualized tracking error** of **0.277**. The **basket** ((DIS, CMCSA, NYT, IPG, OMC)) reaches a correlation of **0.665** with an annualized tracking error of **0.271**.

The rolling TE (60d) is very close: the single has a **p95 TE** of **0.032** (max **0.074**), while the basket has a **p95 TE** of **0.029** (max **0.074**).

Tracking error is similar for the single stock and the basket, with only marginal improvement from diversification. This suggests that sector dynamics are already largely captured by the representative stock in this configuration.


## Consumer Cyclical

In the UMAP, the sector splits into **2 clusters** over **277 stocks**. The structure is very concentrated: the largest cluster (**Cluster 1 (265 stocks)**) contains **265 stocks**, i.e. **95.7%** of the sector.

On the index comparison, the **single representative** (**IP**) reaches a return correlation to the proxy of **0.583** with an **annualized tracking error** of **0.295**. The **basket** ((IP, MCD, GT, F, HD)) increases correlation to **0.786** and reduces annualized tracking error to **0.226**.

The rolling TE (60d) confirms this: the single has a **p95 TE** of **0.060** (max **0.370**), while the basket has a **p95 TE** of **0.054** (max **0.368**).

The Tracking error highlights large and volatile deviations for a single stock, reflecting strong regime dependence. Although a basket improves representativeness, residual deviations remain high, indicating a highly diversified and cyclical sector where single-stock exposure is inadequate.


## Consumer Defensive

In the UMAP, the sector splits into **2 clusters** over **116 stocks**. The structure is strongly concentrated: the largest cluster (**Cluster 1 (105 stocks)**) contains **105 stocks**, i.e. **90.5%** of the sector.

On the index comparison, the **single representative** (**PG**) reaches a return correlation to the proxy of **0.598** with an **annualized tracking error** of **0.207**. The **basket** ((PG, WMT, PEP, TGT, CL)) increases correlation to **0.789** and reduces annualized tracking error to **0.152**.

The rolling TE (60d) confirms this: the single has a **p95 TE** of **0.029** (max **0.104**), while the basket has a **p95 TE** of **0.025** (max **0.105**).

The single stock performs reasonably well, but the basket still provides lower and more stable tracking error. This suggests limited but non-negligible internal diversification: a single stock is usable, but diversification improves robustness.


## Energy

In the UMAP, the sector splits into **3 clusters** over **133 stocks**. The structure is concentrated: the largest cluster (**Cluster 1 (118 stocks)**) contains **118 stocks**, i.e. **88.7%** of the sector.

On the index comparison, the **single representative** (**XOM**) reaches a return correlation to the proxy of **0.725** with an **annualized tracking error** of **0.169**. The **basket** ((XOM, CVX, BP, SLB, NBR)) reaches a correlation of **0.751** with an annualized tracking error of **0.168**.

The rolling TE (60d) is nearly unchanged: the single has a **p95 TE** of **0.026** (max **0.078**), while the basket has a **p95 TE** of **0.027** (max **0.074**).

Tracking error shows very similar behavior for the single stock and the basket. This indicates that Energy is dominated by a common factor, making a well-chosen single stock sufficient to represent the sector.


## Financial Services

In the UMAP, the sector splits into **6 clusters** over **598 stocks**. The structure is concentrated but less collapsed than many other sectors: the largest cluster (**Cluster 1 (508 stocks)**) contains **508 stocks**, i.e. **84.9%** of the sector.

On the index comparison, the **single representative** (**WFC**) reaches a return correlation to the proxy of **0.545** with an **annualized tracking error** of **0.263**. The **basket** ((WFC, C, MMC, CNA, PNC)) increases correlation to **0.696** and reduces annualized tracking error to **0.178**.

The rolling TE (60d) confirms this: the single has a **p95 TE** of **0.032** (max **0.114**), while the basket has a **p95 TE** of **0.023** (max **0.113**).

Tracking error remains elevated for the single stock, especially during stress periods, while the basket consistently lowers deviations. This confirms strong internal diversification within the sector and the need for a basket to represent overall financial sector dynamics.


## Healthcare

In the UMAP, the sector splits into **2 clusters** over **442 stocks**. The structure is extremely concentrated: the largest cluster (**Cluster 1 (432 stocks)**) contains **432 stocks**, i.e. **97.7%** of the sector.

On the index comparison, the **single representative** (**JNJ**) reaches a return correlation to the proxy of **0.605** with an **annualized tracking error** of **0.284**. The **basket** ((JNJ, MRK, PFE, LLY, MDT)) increases correlation to **0.800** and reduces annualized tracking error to **0.199**.

The rolling TE (60d) confirms this: the single has a **p95 TE** of **0.043** (max **0.074**), while the basket has a **p95 TE** of **0.034** (max **0.073**).

The single stock exhibits systematically higher tracking error than the basket. This shows that Healthcare is not well represented by one name and that diversification across subsectors is necessary to approximate the sector proxy.


## Industrials

In the UMAP, the sector splits into **2 clusters** over **307 stocks**. The structure is very concentrated: the largest cluster (**Cluster 1 (295 stocks)**) contains **295 stocks**, i.e. **96.1%** of the sector.

On the index comparison, the **single representative** (**GE**) reaches a return correlation to the proxy of **0.531** with an **annualized tracking error** of **0.171**. The **basket** ((GE, BA, MMM, HON, EMR)) increases correlation to **0.591** and reduces annualized tracking error to **0.131**.

The rolling TE (60d) confirms lower typical deviations: the single has a **p95 TE** of **0.025** (max **0.236**), while the basket has a **p95 TE** of **0.018** (max **0.236**).

The tracking error of the single stock remains consistently higher than that of the basket, although differences are moderate. This reflects a sector with shared dynamics but meaningful internal diversification, for which a basket offers more reliable representation.


## Real Estate

In the UMAP, the sector splits into **2 clusters** over **152 stocks**. The structure is concentrated: the largest cluster (**Cluster 1 (134 stocks)**) contains **134 stocks**, i.e. **88.2%** of the sector.

On the index comparison, the **single representative** (**WY**) reaches a return correlation to the proxy of **0.431** with an **annualized tracking error** of **0.243**. The **basket** ((WY, HST, FRT, VNO, PCH)) increases correlation to **0.578** and reduces annualized tracking error to **0.173**.

The rolling TE (60d) confirms the reduction in typical deviations: the single has a **p95 TE** of **0.041** (max **0.199**), while the basket has a **p95 TE** of **0.032** (max **0.199**).

The tracking error clearly favors the basket over the single stock, which deviates strongly from the proxy. This reflects heterogeneous sensitivities across real estate subsectors, requiring diversification to represent the sector.


## Technology

In the UMAP, the sector splits into **2 clusters** over **325 stocks**. The structure is extremely concentrated: the largest cluster (**Cluster 1 (316 stocks)**) contains **316 stocks**, i.e. **97.2%** of the sector.

On the index comparison, the **single representative** (**HPQ**) reaches a return correlation to the proxy of **0.612** with an **annualized tracking error** of **0.328**. The **basket** ((HPQ, IBM, INTC, AAPL, TER)) increases correlation to **0.802** and reduces annualized tracking error to **0.228**.

The rolling TE (60d) confirms this: the single has a **p95 TE** of **0.033** (max **0.078**), while the basket has a **p95 TE** of **0.023** (max **0.059**).

The tracking error confirms that a single stock cannot stably represent the sector: deviations remain large and persistent relative to the proxy. A small basket significantly reduces normalized tracking error, indicating that Technology is diversified internally and requires intra-sector diversification to capture its aggregate behavior.



## Utilities

In the UMAP, the sector splits into **3 clusters** over **28 stocks**. The structure is the least concentrated among the sectors in this export: the largest cluster (**Cluster 1 (16 stocks)**) contains **16 stocks**, i.e. **57.1%** of the sector.

On the index comparison, the **single representative** (**ED**) reaches a return correlation to the proxy of **0.549** with an **annualized tracking error** of **0.206**. The **basket** ((ED, DTE, AEP, CNP, NEE)) increases correlation to **0.673** and reduces annualized tracking error to **0.170**.

The rolling TE (60d) confirms improved typical tracking: the single has a **p95 TE** of **0.026** (max **0.069**), while the basket has a **p95 TE** of **0.020** (max **0.070**).

The basket consistently lowers tracking error relative to the single stock. Although Utilities are relatively defensive, the results show sufficient internal diversification that a basket provides a more faithful sector representation.

# Chapter 2 — Results  
**Sector Proxy vs Stock Representativeness**

Here we ask a very practical question: if an investor wants “sector exposure”, is it enough to hold one representative stock, or does one need a small basket to avoid idiosyncratic risk? Instead of assuming that sectors are homogeneous by definition, we test this directly from the data using the exact pipeline implemented in our Chapter 2 module: we build a sector proxy from weighted stock returns, filter out illiquid names, score representativeness with rolling correlations, select candidates with a lifetime-aware ranking rule, and validate the result using tracking error through time.

---

## 2.1 Intra-Sector Structure: How Cohesive Are Sectors?

We start from structure, because representativeness is only plausible if stocks inside a sector actually share common dynamics. For each sector, we build a stock–stock correlation matrix using **pairwise overlap** (each pair uses all dates they share, rather than forcing a common intersection across all tickers). This matters in real equity data because lifetimes differ: if we required a common overlap, we would either lose most of the history or drop many stocks.

We then convert correlations into distances and apply **UMAP** on this precomputed distance matrix to obtain a 2D map of intra-sector behavior. In parallel, we cluster stocks using **Agglomerative Clustering** on the same distance matrix, with a complete-linkage rule and a data-driven threshold (by default, a percentile of the empirical distance distribution). Stocks with too much missing pairwise overlap are explicitly handled and assigned to a dedicated “Missing data” label rather than silently contaminating the clustering.

Structurally, most sectors display a strong collapse into one dominant cluster. Technology is a striking example with **2 clusters over 325 stocks** and a main cluster containing **316 stocks (97.2%)**. Healthcare is even more concentrated (**432 of 442 stocks, 97.7%**) and Consumer Cyclical also collapses heavily (**265 of 277 stocks, 95.7%**). This kind of collapse suggests that common dynamics exist and that simplified representation might be feasible. Some sectors, however, already signal internal diversification at the structural level. Financial Services shows richer segmentation with **6 clusters over 598 stocks**, and Basic Materials is less collapsed than most sectors with **2 clusters over 128 stocks** and a main cluster containing **94 stocks (73.4%)**, hinting at distinct behaviors across materials and value chains. Utilities also stands out despite its smaller universe: **3 clusters over 28 stocks** with the largest containing **16 stocks (57.1%)**, indicating that even defensive sectors may host multiple stable “behavioral types”.

This first step does not yet prove whether one stock can track the sector, but it tells us where simplification is structurally plausible—and where diversification is already justified before looking at performance.

---

## 2.2 From Structure to Candidates: Which Stocks Look Representative?

Once structure is mapped, we shift from “how the sector is organized” to “which individual names best approximate the sector proxy”. In our pipeline, this is done by computing **rolling correlations** between each stock’s daily returns and the **sector proxy** over a trailing window (by default 60 trading days, with a minimum number of observations to avoid unstable estimates). The representativeness score of a stock is simply the **average** of this rolling correlation through time, so a stock ranks well only if it stays consistently aligned with the proxy rather than occasionally matching it.

At this point, two important implementation details shape the candidates we select. First, we apply a liquidity filter that keeps only stocks with sufficiently frequent nonzero volume and an average volume above a sector threshold, so the candidates are tradable rather than statistically convenient. Second, we account for unequal lifetimes when selecting the Top-5 basket: we compute a **lifetime fraction** for each ticker (how much of the sector panel it actually covers) and use a lifetime-weighted score  
`effective_score = rep_score × (lifetime_fraction^alpha)` (with `alpha = 2` by default). This prevents short-lived tickers from ranking artificially high because they only exist during a “lucky” period. When two candidates are close, volatility acts as a tie-breaker in favor of more stable names.

The outcome of this step is a single top representative stock and a Top-5 list for each sector, selected with a rule that prioritizes persistent proxy alignment while penalizing short histories.

---

## 2.3 Does the Representative Stock Really Track the Sector?

We then test representativeness in the most intuitive way: we plot normalized indices (base = 100) built from daily returns for the **sector proxy**, the **single representative stock**, and the **Top-5 basket**. Indices are computed using compounding consistent with the data (log-safe when needed), so the curves directly show whether an investor holding the single stock (or the basket) would have experienced the same return path as the sector.

In some sectors, the representative behaves like a true sector substitute. Energy is the clearest case: the representative stock **XOM** achieves a return correlation of **0.725** to the sector proxy, while the basket only moves this slightly to **0.751**, and their tracking error behavior is nearly indistinguishable. In this configuration, sector dynamics are dominated by a strong common factor, so a single well-chosen stock is enough to reproduce the sector’s return path.

Other sectors show the opposite pattern: the representative tracks the proxy on average but drifts meaningfully at times, and the basket visibly stabilizes the match. Technology illustrates this strongly: **HPQ** alone reaches correlation **0.612** with a large annualized tracking deviation (**0.328**), whereas the basket **(HPQ, IBM, INTC, AAPL, TER)** rises to correlation **0.802** and reduces the annualized deviation to **0.228**. Healthcare shows the same logic: **JNJ** alone correlates **0.605** with the proxy and deviates substantially, while the basket **(JNJ, MRK, PFE, LLY, MDT)** climbs to **0.800** and reduces deviation from **0.284** to **0.199**.

There are also sectors where the basket offers only marginal improvement over the best single name in our selection. Communication Services is the main example here: **DIS** correlates **0.674** with the proxy and the basket **(DIS, CMCSA, NYT, IPG, OMC)** is essentially the same (**0.665**), with very similar deviation magnitude (**0.277** vs **0.271**). In this configuration, the chosen representative already captures most sector dynamics, so diversification adds little incremental tracking benefit.

By this stage, the core tension is visible: structural cohesion does not guarantee return-path fidelity, and the basket often matters precisely when idiosyncratic exposures start to dominate.

---

## 2.4 Tracking Error: When Simplification Breaks Down

To validate the conclusion rigorously, we move from return-path inspection to a risk-based measure: **tracking error through time**. In our pipeline, tracking error is computed exactly as the rolling standard deviation of active returns:  
`TE_t = rolling_std( r_candidate(t) − r_sector_proxy(t) )`,  
with a 60-day window (and optionally 120-day). This is what appears in the tracking-error plots. The tracking error axis is normalized, which makes comparisons across sectors more readable without changing the underlying logic.

The pattern is consistent with the index evidence. Energy again shows that simplification works: single and basket TE curves sit on top of each other, confirming that the one-stock solution remains stable through time. Communication Services also shows close behavior, with only a slight reduction in TE for the basket (p95 is **0.032** for the single vs **0.029** for the basket, with identical max around **0.074**), consistent with the limited incremental value of diversification in this configuration.

In sectors where we observed persistent deviations, tracking error provides the decisive validation. Basic Materials is a textbook case: its internal structure is less collapsed (largest cluster only **73.4%**), and the representative stock **AA** displays substantially higher tracking error than the basket. The improvement is not cosmetic: the basket reduces both typical deviations (p95 **0.021** vs **0.028**) and extreme deviations (max **0.035** vs **0.047**), consistent with internal diversification across different materials exposures and value chains. Technology shows the same principle: the basket materially reduces tracking error (p95 **0.023** vs **0.033**, max **0.059** vs **0.078**), confirming that a single representative stock cannot stably represent the sector.

Some sectors are simply harder because their deviations are regime-driven. Consumer Cyclical illustrates this: although the sector appears structurally very concentrated, tracking error for the representative **IP** shows large and volatile spikes (p95 **0.060**, max **0.370**), and while the basket reduces typical deviation slightly (p95 **0.054**), extremes remain very large. This is exactly what “cyclical sector diversification” means in practice: even the best candidate stock remains sensitive to idiosyncratic shocks and macro transitions, and diversification helps but cannot fully eliminate stress episodes.

Finally, several sectors fall into an intermediate zone where a single stock is “usable” but not optimal. Consumer Defensive (PG) is representative of this: the single stock already captures a large fraction of proxy behavior, yet the basket systematically lowers and stabilizes tracking error (p95 **0.029** → **0.025**). Industrials behaves similarly, with moderate improvements in alignment and reduced typical tracking error when moving from **GE** to the Top-5 basket.

---

## 2.5 Sector-Level Takeaways

The results lead to a practical classification rooted in the metrics we actually observe:

Energy is the clearest example where a single stock can represent the sector reliably, because both the return path and tracking error are essentially unchanged when moving from the single representative to the basket. Communication Services is close to this “single-stock works” regime in our configuration, since diversification brings only marginal tracking improvement.

Technology, Healthcare, Financial Services, Industrials, Consumer Defensive, Utilities, and Basic Materials sit in the “basket safer” regime: a single stock captures the sector direction on average, but the basket consistently improves correlation and reduces tracking error, especially in unstable periods. Basic Materials in particular demonstrates that when sector structure is less collapsed, diversification becomes a measurable necessity.

Consumer Cyclical stands out as a sector where simplification breaks down: the representative stock shows frequent and extreme tracking-error spikes, and although a basket improves typical tracking, residual deviations remain large, reflecting genuine regime dependence and internal diversification that cannot be compressed into a single name.

---

## 2.6 What This Means for Investors

The investor takeaway is that sector simplification is real, but conditional. A sector can look structurally cohesive in UMAP space and still exhibit substantial idiosyncratic risk once we test real proxy tracking. Representativeness is not about choosing a famous ticker; in our pipeline it emerges from persistent rolling correlation to the sector proxy, filtered for liquidity and penalized for short lifetimes. Once representativeness is defined correctly, tracking error becomes the arbiter: if the single stock’s tracking error stays close to the basket’s through time, simplification is justified; if it does not, diversification is not optional.

In practice, the Top-5 basket offers an attractive compromise. Because it is selected from behavior (rolling correlations), adjusted for lifetime coverage, and built with equal-weighting that naturally handles missing returns without artificially shrinking the basket, it captures most of the diversification benefit while keeping implementation simple. The data therefore supports a selective approach: use a single-stock proxy only in sectors where the evidence shows it behaves like the sector, and rely on a small basket in sectors where internal diversification expresses itself as persistent tracking error.
